# Comparación de Modelos

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.scorer import VolatilityScorer
from src.utils import test_toolkit as ts
from src.utils import ts_toolkit as tst

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results"

csv_paths = [
    RESULTS_PATH / "garch" / "best_garch_predictions.csv",
    RESULTS_PATH / "egarch" / "best_egarch_predictions.csv",
    RESULTS_PATH / "har" / "best_har_predictions.csv",
    RESULTS_PATH / "wtd_har" / "best_wtd_har_predictions.csv",
    RESULTS_PATH / "gru" / "exec_2" / "gru_test_predictions.csv",
    RESULTS_PATH / "tcn" / "exec_6" / "tcn_test_predictions.csv",
    RESULTS_PATH / "bigru_tcn" / "exec_1" / "bigru_tcn_test_predictions.csv",
    RESULTS_PATH / "causal_transformer" / "exec_2" / "causal_transformer_test_predictions.csv",
    RESULTS_PATH / "band_causal_tcn" / "exec_1" / "band_causal_tcn_test_predictions.csv",
]

model_names = [
    "GARCH",
    "EGARCH",
    "HAR",
    "WTD-HAR",
    "GRU",
    "TCN",
    "BiGRU-TCN",
    "Causal Transformer",
    "Band Causal TCN"
]

test_bundle = ts.load_test_predictions_matrix(
    csv_paths=csv_paths,
    model_names=model_names,
)

dates_test = test_bundle["dates"]
y_true_test = test_bundle["y_true"]
preds_matrix = test_bundle["preds_matrix"]
models_loaded = test_bundle["model_names"]

## Introducción

Esta libreta se dedica a la comparación conjunta del desempeño predictivo fuera de muestra de los modelos entrenados y evaluados para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. A diferencia de las libretas anteriores, donde cada modelo se formuló, calibró y diagnosticó de manera individual tanto dentro como fuera de la muestra, aquí se alinean temporalmente las series pronosticadas sobre el conjunto de prueba para contrastar directamente las distintas arquitecturas bajo las mismas condiciones de información.

El análisis integra métricas puntuales y probabilísticas de evaluación, pruebas de comparación predictiva por pares mediante la prueba de Diebold-Mariano con *bootstrap* de bloques estacionarios, medidas de magnitud del efecto a través del delta de Cliff y procedimientos de selección del conjunto superior de modelos (*Model Confidence Set*) sobre la pérdida **QLIKE**. Este marco comparativo permite evaluar de forma rigurosa si las diferencias de desempeño entre modelos reflejan ventajas predictivas consistentes o variaciones asociadas a los regímenes de volatilidad del mercado.

## Resultados principales

En la comparación fuera de muestra, *TCN* encabezó el desempeño en la métrica principal (`QLIKE`), en las métricas heterogéneas (`HMSE` y `HMAE`) y en `CRPS`, seguido de cerca en `QLIKE` por *Causal Transformer* y *Band Causal TCN* para conformar el grupo de los tres primeros modelos. Estos tres enfoques mantuvieron valores bajos y competitivos tanto en las pérdidas cuantilares de Pinball como en `RMSE`, superando en `QLIKE` y `HMSE` a referencias econométricas como *WTD-HAR*, *GARCH* y *HAR*, y marcando una ventaja clara frente a *EGARCH*, *GRU* y *BiGRU-TCN*, donde se observaron incrementos pronunciados en `QLIKE` y `HMSE`.

In [3]:
ts.display_models_comparison(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metrics=("qlike", "crps", "pinball_05", "pinball_95", "rmse", "hmse", "hmae"),
    sort_by="qlike",
    decimals=4,
)

Modelo,QLIKE,CRPS,Pinball (τ=0.05),Pinball (τ=0.95),RMSE,HMSE,HMAE
TCN,0.3675,0.0123,0.0027,0.0015,0.0022,1.7030,0.6482
Causal Transformer,0.3936,0.0123,0.0027,0.0016,0.0021,2.1469,0.6853
Band Causal TCN,0.4022,0.0124,0.0027,0.0016,0.0023,2.0900,0.6927
WTD-HAR,0.4084,0.0124,0.0026,0.0015,0.0022,3.1898,0.7317
GARCH,0.4097,0.0123,0.0028,0.0016,0.0022,2.5417,0.6710
HAR,0.4471,0.0124,0.0026,0.0016,0.0022,3.3125,0.7970
EGARCH,0.5068,0.0123,0.0028,0.0017,0.0022,10.1008,0.7719
GRU,0.5109,0.0124,0.0027,0.0017,0.0020,6.4425,0.8131
BiGRU-TCN,0.6402,0.0126,0.0026,0.0020,0.0021,10.1471,1.0021


Para evaluar la significancia estadística y la magnitud del efecto entre pares de modelos sobre la pérdida `QLIKE`, se aplicó la prueba de Diebold-Mariano con *bootstrap* de bloques (`n_bootstrap = 1000`, $\alpha = 0.05$) y el delta de Cliff. Este procedimiento permitió obtener intervalos de confianza *bootstrap* para las diferencias de pérdida y contrastar si excluyen el cero junto con la dirección y magnitud práctica de cada comparación por pares.

In [4]:
dm_results = ts.diebold_mariano_test(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metric="qlike",
    alpha=0.05,
    n_bootstrap=1000,
    seed=SEED,
)

cliff_results = ts.cliffs_delta(
    y_true=y_true_test,
    preds_matrix=preds_matrix,
    model_names=models_loaded,
    metric="qlike",
)

Al contrastar los tres modelos con menor `QLIKE` (*TCN*, *Causal Transformer* y *Band Causal TCN*) a partir del mapa de calor del delta de Cliff condicionado por los intervalos de confianza de Diebold-Mariano, se observaron magnitudes de efecto muy pequeñas entre ellos (`-0.03` y `-0.02` en la fila de *TCN* frente a *Causal Transformer* y *Band Causal TCN*, y `0.01` entre estos dos últimos). En particular, las celdas correspondientes a *TCN* frente a *Causal Transformer* y *Band Causal TCN* no aparecieron resaltadas como estadísticamente destacadas bajo el criterio del intervalo de confianza de Diebold-Mariano, y el cruce entre *Causal Transformer* y *Band Causal TCN* tampoco mostró una diferencia destacada.

En consecuencia, no se sostiene un ganador claro ni una superioridad estadística contundente de *TCN* sobre *Causal Transformer* o *Band Causal TCN* en términos de magnitud del efecto. Aunque *TCN* registró el menor valor puntual en `QLIKE`, frente a estos dos modelos las diferencias en magnitud práctica fueron reducidas y sin significancia estadistica, lo que sugiere un desempeño cercano o en paridad práctica entre los tres enfoques punteros.

In [5]:
ts.plot_cliffs_dm_heatmap(
    cliff_results=cliff_results,
    dm_results=dm_results,
    model_order=models_loaded,
    title="Delta de Cliff por Pares (Referencia DM — QLIKE)",
    fmt="svg",
)